# Notebook to evaluate initial code

In [ ]:
from pfns.scripts.tabpfn_interface import TabPFNClassifier
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


datasets = {
    "iris": load_iris,
    "wine": load_wine,
    "breast_cancer": load_breast_cancer
}

MAX_FEATURES = 25

base_path = "/home/david/ICL-Architectures/PFNs/models_diff/large_config_v3.pt/tabpfn_prior_config_very_large_0_no_seed/"

clf = TabPFNClassifier(
    base_path=base_path,
    device="cuda:0",
    model_string="checkpoint.pt",
    N_ensemble_configurations=32,
)

for name, loader in datasets.items():
    X, y = loader(return_X_y=True)
    X = X[:, :MAX_FEATURES]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    clf_tabpfn = TabPFNClassifier(
        base_path=base_path,
        device="cuda:0",
        model_string="checkpoint.pt",
        N_ensemble_configurations=32,
    )
    clf_tabpfn.fit(X_train, y_train)
    preds_tabpfn = clf_tabpfn.predict(X_test, return_winning_probability=False)
    acc_tabpfn = accuracy_score(y_test, preds_tabpfn)

    rf = RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
    )
    rf.fit(X_train, y_train)
    preds_rf = rf.predict(X_test)
    acc_rf = accuracy_score(y_test, preds_rf)

    print(f"{name:>12} | TabPFN: {acc_tabpfn:.3f} | RF: {acc_rf:.3f}")